## Kalman filter for options

In [ ]:
%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.massive.client
%aimport quantdata.massive.processing
%aimport quantdata.massive.options.contracts

In [ ]:



from datetime import date
from cdxcore.jcpool import JCPool
from cdxcore.pretty import PrettyObject
from cdxcore.subdir import SubDir
from cdxcore.util import fmt_list
from cdxcore.uniquehash import named_unique_filename48_8
import polars as pl

from quantdata.biz_calendar import prev_business_day, to_date
from quantdata.massive.client import MassiveClient
from quantdata.pool import PoolConfig
from quantdata.massive.options.contracts import ContractsFilter, filter_contracts, REJECTED
from quantdata.massive.processing import align_trades_to_quotes, align_quotes_to_trades

from cdxcore.version import version
from cdxcore.verbose import Context
from dataclasses import asdict, dataclass, field
from cdxcore.uniquehash import unique_hash8 
import numpy as np
import math

from quantdata.ncde.implied.__day import _ncde_implied_day_root as _cache
from IPython.display import display, Markdown

from quantdata.massive.processing import filter_valid_quotes
from quantdata.massive.client import MassiveClient

opt_ticker = "O:SPY250123C00614000"
ticker = "SPY"
d = to_date("2025-01-23")
client = MassiveClient()
display( client.fetch_quotes("O:SPY250623P00613000", d) )
display( client.has_any_quotes("O:SPY250623P00613000", d) )
display( client.has_any_trades("O:SPY250623P00613000", d) )
display( filter_valid_quotes( "O:SPY250623P00613000", d) )
display( align_trades_to_quotes( "O:SPY250623P00613000", d, within_session=True, drop_unmatched=False ) )


In [ ]:
from quantdata.massive.processing import (
    TradeQuoteAlignmentConfig,
    filter_valid_trades,
    filter_valid_quotes,
    align_trades_to_quotes,
)

tkr = "O:SPY250623P00613000"

raw_trades = MassiveClient().fetch_trades(tkr, d)
raw_quotes = MassiveClient().fetch_quotes(tkr, d)
valid_trades = filter_valid_trades(tkr, d)
valid_quotes = filter_valid_quotes(tkr, d)

cfg = TradeQuoteAlignmentConfig(drop_unmatched=False)
aligned_keep_all = align_trades_to_quotes(tkr, d, config=cfg)
aligned_matched = aligned_keep_all.filter(pl.col("trade_ts_ns").is_not_null())

print("raw trades:", len(raw_trades))
print("valid trades:", len(valid_trades))
print("raw quotes:", len(raw_quotes))
print("valid quotes:", len(valid_quotes))
print("aligned (keep unmatched):", len(aligned_keep_all))
print("aligned matched only:", len(aligned_matched))

if len(valid_trades) > 0 and len(valid_quotes) > 0:
    print("trade price range:", float(valid_trades["price"].min()), float(valid_trades["price"].max()))
    print("quote bid/ask range:", float(valid_quotes["bid_price"].min()), float(valid_quotes["ask_price"].max()))
    display(aligned_keep_all.select(["ts_ns", "bid_price", "ask_price", "trade_ts_ns", "price"]).head(10))

In [ ]:
# timing vs price-debug for the single valid trade
vt = filter_valid_trades(tkr, d).sort("ts_ns")
vq = filter_valid_quotes(tkr, d).sort("ts_ns")

if len(vt) > 0:
    tts = vt["ts_ns"][0]
    tpx = vt["price"][0]
    around = vq.filter((pl.col("ts_ns") >= tts - pl.duration(seconds=5)) & (pl.col("ts_ns") <= tts + pl.duration(seconds=5)))
    in_spread = around.filter((pl.col("bid_price") <= tpx) & (tpx <= pl.col("ask_price")))
    print("trade ts:", tts, "trade px:", tpx)
    print("quotes within +/-5s:", len(around), "in-spread within +/-5s:", len(in_spread))
    if len(in_spread) > 0:
        nearest = in_spread.with_columns((pl.col("ts_ns") - pl.lit(tts)).alias("dt")).with_columns(pl.col("dt").abs().alias("abs_dt")).sort("abs_dt").head(5)
        display(nearest.select(["ts_ns", "bid_price", "ask_price", "dt", "abs_dt"]))

In [ ]:
from quantdata.massive.options.contracts import filter_contracts, ContractsFilter, REJECTED, PoolConfig, Context
import polars as pl

opt_ticker = "O:SPY250123C00614000"
ticker = "SPY"
d = to_date("2025-01-23")

df = filter_contracts( ticker, d=d, exclude_half_days=True, drop_no_trades_today=True,
                        contracts_filter=ContractsFilter(), pool_config=PoolConfig(), verbose=Context.all )

df.filter(pl.col("ticker") == "O:SPY250623P00613000")

In [ ]:
df